# 하남교산 도로망-토지이용계획도 매핑 가능성 점검

본 절에서는 다음 세 데이터를 활용하여  
하남교산의 도로망과 토지이용계획도의 연계 가능성을 확인한다.

- `23._토지이용계획도_(하남교산).geojson`
- `08.상세도로망_네트워크.geojson`
- `25._교통데이터_코드표.csv`

주요 확인 사항은 다음과 같다.

1. 도로망 데이터에 어떤 코드 컬럼이 존재하는가  
2. 코드표와 실제 도로망 속성이 연결 가능한가  
3. 하남교산 토지이용계획도와 도로망을 공간적으로 매핑할 수 있는가  
4. 어떤 토지이용 구간에 어떤 도로 유형이 배치되는지 확인 가능한가

그리고 최종 목표는 다음과 같다. (해결내용 3번째)
`스마트 안전 교통시설물 설치에 따른 교통량 변화 및 사고율 변화 효과를 계량적으로 산출하고, 이를 반영하여 최적의 시설물 위치를 제시해야 합니다.` 

In [17]:
import pandas as pd
import geopandas as gpd
import numpy as np

# -----------------------------
# 0. 원본 복사
# -----------------------------
road_fix = road.copy()
landuse_fix = landuse.copy()
codebook_fix = codebook.copy()

# -----------------------------
# 1. 코드 정규화 함수
# -----------------------------
def normalize_code(x):
    if pd.isna(x):
        return np.nan
    try:
        # 107, 107.0 -> "107"
        return str(int(float(x)))
    except:
        return str(x).strip()

# -----------------------------
# 2. codebook 정규화
# -----------------------------
codebook_fix["code_norm"] = codebook_fix["code"].apply(normalize_code)

print("=== codebook preview ===")
display(codebook_fix.head())

# -----------------------------
# 3. 도로망 컬럼 중 코드표와 연결 가능한 컬럼 찾기
# -----------------------------
code_cols = sorted(codebook_fix["column_en"].dropna().unique().tolist())
road_cols = road_fix.columns.tolist()

matched_cols = [c for c in code_cols if c in road_cols]
print("=== matched code columns ===")
print(matched_cols)

# -----------------------------
# 4. 각 컬럼 코드 정규화 후 라벨 붙이기
# -----------------------------
road_decoded = road_fix.copy()

for col in matched_cols:
    road_decoded[col + "_norm"] = road_decoded[col].apply(normalize_code)

    temp_map = (
        codebook_fix.loc[codebook_fix["column_en"] == col, ["code_norm", "details"]]
        .drop_duplicates()
        .rename(columns={"details": f"{col}_label"})
    )

    road_decoded = road_decoded.merge(
        temp_map,
        left_on=col + "_norm",
        right_on="code_norm",
        how="left"
    ).drop(columns=["code_norm"], errors="ignore")

print("=== decoded columns added ===")
print([c for c in road_decoded.columns if c.endswith("_label")])

display(
    road_decoded[
        [c for c in ["road_rank", "road_rank_norm", "road_rank_label",
                     "road_type", "road_type_norm", "road_type_label",
                     "link_type", "link_type_norm", "link_type_label",
                     "oneway", "oneway_norm", "oneway_label"]
         if c in road_decoded.columns]
    ].head(10)
)

# -----------------------------
# 5. 좌표계 통일
# -----------------------------
road_decoded = road_decoded.to_crs(landuse_fix.crs)

# -----------------------------
# 6. 하남교산과 intersect 하는 도로만 추출
# -----------------------------
road_decoded = road_decoded.drop(columns=[c for c in ["index_left", "index_right"] if c in road_decoded.columns], errors="ignore")
landuse_fix = landuse_fix.drop(columns=[c for c in ["index_left", "index_right"] if c in landuse_fix.columns], errors="ignore")

road_gyosan = gpd.sjoin(
    road_decoded,
    landuse_fix[["blockType", "geometry"]],
    how="inner",
    predicate="intersects"
)

road_gyosan = road_gyosan.drop(columns=["index_right"], errors="ignore")

print("전체 도로 수:", len(road_decoded))
print("하남교산과 intersect 하는 도로 수:", len(road_gyosan))

display(
    road_gyosan[
        [c for c in ["road_name", "road_rank", "road_rank_label",
                     "road_type", "road_type_label",
                     "link_type", "link_type_label",
                     "oneway", "oneway_label",
                     "blockType", "geometry"]
         if c in road_gyosan.columns]
    ].head(10)
)

# -----------------------------
# 7. 정말 라벨이 붙었는지 확인
# -----------------------------
print("=== road_rank_label value counts ===")
display(road_gyosan["road_rank_label"].value_counts(dropna=False).head(20))

print("=== road_type_label value counts ===")
display(road_gyosan["road_type_label"].value_counts(dropna=False).head(20))

print("=== link_type_label value counts ===")
display(road_gyosan["link_type_label"].value_counts(dropna=False).head(20))

print("=== oneway_label value counts ===")
display(road_gyosan["oneway_label"].value_counts(dropna=False).head(20))

# -----------------------------
# 8. 토지이용별 도로 유형 분포 확인
# -----------------------------
if "road_type_label" in road_gyosan.columns:
    print("=== blockType x road_type_label ===")
    display(
        pd.crosstab(
            road_gyosan["blockType"],
            road_gyosan["road_type_label"],
            normalize="index"
        ).round(3)
    )

if "road_rank_label" in road_gyosan.columns:
    print("=== blockType x road_rank_label ===")
    display(
        pd.crosstab(
            road_gyosan["blockType"],
            road_gyosan["road_rank_label"],
            normalize="index"
        ).round(3)
    )

# -----------------------------
# 9. 해석 가능 여부 체크
# -----------------------------
result_check = {
    "matched_code_columns": matched_cols,
    "decoded_road_rank_labels": road_gyosan["road_rank_label"].notna().sum() if "road_rank_label" in road_gyosan.columns else 0,
    "decoded_road_type_labels": road_gyosan["road_type_label"].notna().sum() if "road_type_label" in road_gyosan.columns else 0,
    "decoded_link_type_labels": road_gyosan["link_type_label"].notna().sum() if "link_type_label" in road_gyosan.columns else 0,
    "decoded_oneway_labels": road_gyosan["oneway_label"].notna().sum() if "oneway_label" in road_gyosan.columns else 0,
    "can_use_for_facility_logic": True
}

print("=== result_check ===")
print(result_check)

=== codebook preview ===


,column_en,column_ko,code,details,remarks,code_norm
0,road_rank,도로의 등급,101.0,고속국도,NaN,101
1,road_rank,도로의 등급,102.0,도시고속국도,NaN,102
2,road_rank,도로의 등급,103.0,일반국도,NaN,103
3,road_rank,도로의 등급,104.0,특별광역시도,NaN,104
4,road_rank,도로의 등급,105.0,국가지원지방도,NaN,105


=== matched code columns ===
['link_type', 'oneway', 'pavement', 'rc_hist', 'road_rank', 'road_type']
=== decoded columns added ===
['link_type_label', 'oneway_label', 'pavement_label', 'rc_hist_label', 'road_rank_label', 'road_type_label']


,road_rank,road_rank_norm,road_rank_label,road_type,road_type_norm,road_type_label,link_type,link_type_norm,link_type_label,oneway,oneway_norm,oneway_label
0,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
1,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
2,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
3,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
4,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
5,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
6,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
7,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
8,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)
9,107,107,시군도,0,0,일반도로,32768,32768,비분리,0,0,규제없음(양방통행)


전체 도로 수: 24003
하남교산과 intersect 하는 도로 수: 176


...

=== blockType x road_type_label ===


road_type_label,교량,일반도로
blockType,,
경관녹지,0.000,1.000
공공공지,0.000,1.000
공공청사,0.000,1.000
공동주택,0.000,1.000
공원,0.000,1.000
교육시설,0.000,1.000
근린생활시설용지,0.000,1.000
단독주택,0.000,1.000
도로,0.016,0.984


=== blockType x road_rank_label ===


road_rank_label,시군도,일반국도
blockType,,
경관녹지,1.000,0.000
공공공지,1.000,0.000
공공청사,1.000,0.000
공동주택,1.000,0.000
공원,0.895,0.105
교육시설,1.000,0.000
근린생활시설용지,0.000,1.000
단독주택,0.714,0.286
도로,0.869,0.131


=== result_check ===
{'matched_code_columns': ['link_type', 'oneway', 'pavement', 'rc_hist', 'road_rank', 'road_type'], 'decoded_road_rank_labels': 176, 'decoded_road_type_labels': 176, 'decoded_link_type_labels': 176, 'decoded_oneway_labels': 176, 'can_use_for_facility_logic': True}


In [18]:
print(result_check)

{'matched_code_columns': ['link_type', 'oneway', 'pavement', 'rc_hist', 'road_rank', 'road_type'], 'decoded_road_rank_labels': 176, 'decoded_road_type_labels': 176, 'decoded_link_type_labels': 176, 'decoded_oneway_labels': 176, 'can_use_for_facility_logic': True}


In [19]:
display(road_gyosan[["road_name", "road_rank_label", "road_type_label", "link_type_label", "oneway_label", "blockType"]].head(20))

,road_name,road_rank_label,road_type_label,link_type_label,oneway_label,blockType
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),경관녹지
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공동주택
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공동주택
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),도로
16769,고골로,시군도,일반도로,비분리,규제없음(양방통행),도로
17970,대성로,시군도,일반도로,비분리,규제없음(양방통행),도로
17998,샘재로,시군도,일반도로,비분리,규제없음(양방통행),도로


In [20]:
display(road_gyosan["road_type_label"].value_counts(dropna=False).head(20))

road_type_label
일반도로    174
교량        2
Name: count, dtype: int64

In [21]:
display(pd.crosstab(road_gyosan["blockType"], road_gyosan["road_type_label"], normalize="index").round(3))

road_type_label,교량,일반도로
blockType,,
경관녹지,0.000,1.000
공공공지,0.000,1.000
공공청사,0.000,1.000
공동주택,0.000,1.000
공원,0.000,1.000
교육시설,0.000,1.000
근린생활시설용지,0.000,1.000
단독주택,0.000,1.000
도로,0.016,0.984


## 1. 도로망 데이터와 토지이용계획도의 연계 가능성 분석

본 연구에서는 국가도로망 데이터(링크 기반)와 하남교산 토지이용계획도(폴리곤 기반)를 활용하여  
도로-토지이용 간 공간적 연계를 수행하였다.

### 1.1 코드 기반 도로 속성 해석

도로망 데이터는 road_rank, road_type, link_type, oneway 등의 코드값으로 구성되어 있으며,  
코드북(codebook)을 활용하여 각 도로의 의미를 다음과 같이 해석하였다.

- road_rank: 도로 위계 (시군도, 국도 등)
- road_type: 도로 형태 (일반도로, 교량 등)
- link_type: 연결 구조 (비분리, 교차로 등)
- oneway: 통행 방식 (양방통행, 일방통행)

코드 정규화 과정을 통해 float/int 혼합 문제를 해결하고,  
모든 도로 링크에 대해 의미 기반 라벨을 부여하였다.

---

### 1.2 공간 조인을 통한 도로-토지 매핑

도로(Line)와 토지이용계획도(Polygon)를 `intersects` 기준으로 공간 조인하여  
각 도로가 어떤 토지이용 유형(blockType)에 속하는지 매핑하였다.

그 결과,

- 총 176개의 도로 링크가 하남교산 영역과 공간적으로 연계됨
- 각 도로에 대해 토지이용 유형 + 도로 속성 동시 확보 가능

---

### 1.3 하남교산 도로망 특성 분석

분석 결과, 하남교산 도로망은 다음과 같은 특징을 보인다.

- 대부분 도로가 **시군도(road_rank)**에 해당
- 거의 모든 도로가 **일반도로(road_type)** 구조
- 대부분 **비분리(link_type)** 구조
- 대부분 **양방통행(oneway)**

즉, 하남교산은 고속 간선 중심 구조가 아니라  
**생활권 중심의 일반도로 네트워크**로 구성된 신도시 형태임을 확인할 수 있다.

---

## 2. 도로망 네트워크를 통한 “도로 예측” 가능성

### 2.1 결론: 정확한 예측은 불가능

본 데이터는 현재 시점의 도로망을 기반으로 하며,  
하남교산은 개발 예정지이므로 실제 미래 도로 구조를 직접적으로 예측하는 것은 불가능하다.

특히,

- 도로 개설 계획
- 차로 수 변화
- 교차로 설계 방식
- 신호 체계

와 같은 정보는 포함되어 있지 않기 때문에  
**완전한 네트워크 예측 모델링은 수행할 수 없다.**

---

### 2.2 그러나 “유형 기반 추정”은 가능

비록 정확한 도로를 예측할 수는 없지만,  
다음 정보를 활용하면 **도로 특성 수준의 추정은 가능하다.**

- 기존 도로망의 속성 (road_rank, road_type, link_type)
- 토지이용계획 (blockType)
- 공간적 인접성

이를 기반으로 다음과 같은 해석이 가능하다.

#### 예시

- 공동주택 / 교육시설 인접 도로 → 생활도로 + 보행 중심 도로
- 상업시설 / 근린생활시설 인접 도로 → 교통량 높은 일반도로
- 공원 / 녹지 인접 도로 → 보행자 중심 저속도로

즉, 도로의 **정확한 위치/구조는 예측 불가능하지만**,  
도로의 **기능적 성격(생활도로, 보행 중심, 교통 중심 등)은 추정 가능하다.**

---

## 3. 본 연구에서의 활용 전략

본 분석에서는 도로망을 “예측”의 대상으로 사용하지 않고,  
다음과 같은 역할로 활용한다.

### 3.1 도로망의 역할

- 시설 설치 후보지 필터링
- 도로 유형 기반 위험도 해석
- 토지이용과 결합한 공간적 맥락 제공

### 3.2 위험도 모델의 역할

- 사고 발생 가능성 정량화
- 취약계층(어린이, 노인) 위험 지역 도출

### 3.3 문헌 기반 효과계수(CMF)의 역할

- 시설 설치 시 기대 사고 감소 효과 추정
- 정책적 타당성 확보

---

## 4. 핵심 결론

- 도로망 데이터만으로 미래 도로를 “예측”하는 것은 불가능
- 그러나 토지이용 + 도로 속성을 활용하면  
  **도로의 기능적 유형은 충분히 추정 가능**
- 이를 기반으로 스마트 교통안전시설의 설치 위치를  
  **합리적으로 제안하는 것은 가능**

즉,

`"도로를 예측하는 것이 아니라,  도로의 역할과 위험을 해석하여  시설을 배치하는 것이 본 연구의 핵심이다."`

## 하남교산 도로망-토지이용계획도 연계 분석

본 절에서는 하남교산 토지이용계획도와 도로망 네트워크를 연계하여,  
계획지 내부 및 인접 도로의 기능적 특성을 해석하고 스마트 안전 교통시설물 설치의 기초 자료로 활용하고자 한다.

분석에는 다음 세 자료를 사용하였다.

- `23._토지이용계획도_(하남교산).geojson`
- `08.상세도로망_네트워크.geojson`
- `25._교통데이터_코드표.csv`

특히 도로망 데이터의 코드값(`road_rank`, `road_type`, `link_type`, `oneway`)은  
코드표를 활용해 실제 의미로 변환한 뒤 토지이용계획도와 공간적으로 연결하였다.

## 1. 도로망 데이터 해석 가능성

코드표와 도로망 데이터를 연결한 결과,  
하남교산과 공간적으로 접하는 도로 링크 176개에 대해 다음 속성을 해석할 수 있었다.

- 도로 위계(`road_rank`)
- 도로 형태(`road_type`)
- 링크 구조(`link_type`)
- 통행 방식(`oneway`)

이는 하남교산 계획지에 접하는 도로가 어떤 성격을 가지는지 파악할 수 있음을 의미한다.
즉, 향후 시설 배치 시 단순히 위험 격자만 보는 것이 아니라  
**도로의 기능적 특성과 토지이용 맥락을 함께 반영한 입지 판단**이 가능하다.

## 2. 하남교산 도로망의 주요 특징

분석 결과, 하남교산과 공간적으로 연결되는 도로는 대부분 다음 특성을 보였다.

- `road_rank`: 시군도
- `road_type`: 일반도로
- `link_type`: 비분리
- `oneway`: 규제없음(양방통행)

즉, 하남교산의 도로망은 고속 간선 중심 구조라기보다  
**생활권 중심의 일반도로 네트워크**에 가까운 형태로 해석된다.

또한 토지이용계획도와의 매핑 결과,  
공동주택, 교육시설, 공원, 근린생활시설용지, 보행자전용도로 등  
생활권 기능이 강한 블록과 일반도로가 직접적으로 연결되어 있음을 확인하였다.

이는 하남교산의 교통안전 이슈가  
대규모 통과 교통보다 **생활도로·교차부·보행 활동 구간의 안전 문제**와 더 밀접할 가능성을 시사한다.

## 3. 도로를 "예측"한 것인가?

본 분석은 하남교산의 미래 도로를 직접 예측한 것이 아니다.

현재 활용한 도로망 데이터는 기존 도로 링크와 속성에 대한 자료이며,  
이를 토지이용계획도와 연결해 **계획지 내부 도로의 기능적 성격을 추정**한 것이다.

따라서 본 연구에서 가능한 것은 다음과 같다.

- 공동주택·교육시설 인접 구간은 생활도로/보행활동 구간으로 해석
- 상업시설·근린생활시설 인접 구간은 교통량 유입 가능 구간으로 해석
- 공원·보행자전용도로 인접 구간은 보행 친화형 안전시설 필요 구간으로 해석

반면, 다음과 같은 정보는 본 데이터만으로 직접 예측할 수 없다.

- 실제 교차로 상세 형상
- 차로 수 변경
- 신호 운영 방식
- 향후 개통 후 실교통량 변화

즉, 본 분석은 **미래 도로의 정확한 형상 예측이 아니라, 시설 설치를 위한 기능적 도로 해석**에 해당한다.

## 4. 스마트 안전시설 배치와의 연결

이 단계의 목적은 도로망을 예측하는 것이 아니라,  
**어떤 토지이용 구간의 어떤 도로에 어떤 시설이 적합한지 판단하는 기준을 만드는 것**이다.

예를 들어,

- 공동주택·교육시설·보행자전용도로 인접 일반도로  
  → 어린이 보행 안전시설 우선 검토
- 근린생활시설·상업시설·종합의료시설 인접 일반도로  
  → 노인 생활이동 안전시설 우선 검토
- 도로·하천·교량 구간  
  → 시야 확보, 경고, 속도관리형 시설 검토

따라서 도로망 데이터는
**시설 설치 후보지를 필터링하고, 토지이용과 결합해 시설 유형을 분류하는 근거 자료**로 활용한다.

## 5. 문헌 기반 효과 추정의 필요성

하남교산은 아직 완전히 조성된 도시가 아니므로,  
시설 설치 전후의 시계열 자료를 이용한 직접적인 인과효과 추정은 어렵다.

따라서 본 연구에서는 향후 시설 효과를 정량화할 때,
현장 관측 기반 Before-After 분석 대신  
**문헌 기반 효과계수(CMF)와 검증된 안전 대책(Proven Safety Countermeasures)**를 활용한
시나리오 기반 추정을 적용하고자 한다.

대표적으로 참고 가능한 대책은 다음과 같다.

- **Pedestrian Hybrid Beacon (PHB)**  
  - FHWA Proven Safety Countermeasure로 제시된 보행 횡단 안전 대책
- **Rectangular Rapid Flashing Beacon (RRFB)**  
  - CMF Clearinghouse에서 보행자 사고 감소 효과 제시
- **Road Diets (Roadway Reconfiguration)**  
  - FHWA Proven Safety Countermeasure로 제시된 속도 저감 및 충돌 감소 대책

따라서 다음 단계에서는  
하남교산의 토지이용-도로 기능 유형별로 적합한 시설을 매칭하고,  
문헌 기반 효과계수를 적용하여 기대 사고 감소 효과를 추정한다.

## 참고 문헌 및 근거 자료

- FHWA, **Proven Safety Countermeasures**  
- FHWA, **Pedestrian Hybrid Beacon: Proven Safety Countermeasures Best Practices and Recommendations**  
- FHWA, **Road Diets (Roadway Reconfiguration)**  
- FHWA CMF Clearinghouse, **Crash Modification Factors for Pedestrian Treatments**  
- FHWA, **Toolbox of Pedestrian Countermeasures and Their Potential Effectiveness**

> 위에 적은 PHB, RRFB, Road Diet는 모두 FHWA의 공식 안전 대책/효과 자료에서 확인 가능하다. RRFB는 CMF Clearinghouse에서 보행자 관련 CMF 0.526, 즉 CRF 47.4% 사례가 제시되어 있고, PHB와 Road Diet도 FHWA의 Proven Safety Countermeasure 자료로 안내된다.

In [23]:
summary_table = pd.DataFrame({
    "item": [
        "Matched code columns",
        "Decoded links in Gyosan area",
        "Dominant road rank",
        "Dominant road type",
        "Dominant link type",
        "Dominant traffic direction"
    ],
    "value": [
        ", ".join(result_check["matched_code_columns"]),
        result_check["decoded_road_type_labels"],
        road_gyosan["road_rank_label"].mode().iloc[0] if road_gyosan["road_rank_label"].notna().any() else None,
        road_gyosan["road_type_label"].mode().iloc[0] if road_gyosan["road_type_label"].notna().any() else None,
        road_gyosan["link_type_label"].mode().iloc[0] if road_gyosan["link_type_label"].notna().any() else None,
        road_gyosan["oneway_label"].mode().iloc[0] if road_gyosan["oneway_label"].notna().any() else None,
    ]
})

display(summary_table)

,item,value
0,Matched code columns,"link_type, oneway, pavement, rc_hist, road_ran..."
1,Decoded links in Gyosan area,176
2,Dominant road rank,시군도
3,Dominant road type,일반도로
4,Dominant link type,비분리
5,Dominant traffic direction,규제없음(양방통행)


## 하남교산 스마트 안전 교통시설 후보지 분류

본 절에서는 하남교산 토지이용계획도와 도로망 데이터를 연계하여  
스마트 안전 교통시설물 설치 후보지를 유형별로 분류한다.

본 연구는 하남교산의 미래 도로 형상을 직접 예측하지 않고,  
토지이용계획도와 기존 도로망 속성을 연결하여  
계획지 내부 도로의 기능적 특성을 해석하는 방식으로 접근하였다.

시설 유형은 다음과 같이 구분한다.

- 어린이 보행 안전형: 교육시설, 공동주택, 공원, 보행자전용도로 인접 일반도로
- 노인 생활이동 안전형: 근린생활시설, 상업시설, 종합의료시설, 공공공지 인접 일반도로
- 일반 생활도로 속도관리형: 도로, 주거, 혼합생활권 인접 일반도로

이후 각 후보지에 대해 문헌 기반 효과계수를 적용하여  
시설 설치 시 기대되는 사고위험 저감 효과를 시나리오 방식으로 추정한다.

## 참고 문헌 및 적용 근거

- FHWA Proven Safety Countermeasures
- FHWA STEP (Safe Transportation for Every Pedestrian)
- FHWA Toolbox of Pedestrian Countermeasures
- FHWA CMF Clearinghouse

대표 적용 시설은 다음과 같다.

- PHB (Pedestrian Hybrid Beacon)
- RRFB (Rectangular Rapid Flashing Beacon)
- Speed Safety Camera / Dynamic Speed Management

본 연구에서는 직접적인 사후 평가 데이터가 없으므로,  
문헌에서 제시된 안전효과를 참고하여  
시설별 기대 효과를 시나리오 기반으로 적용한다.

## 1. 하남교산 도로망 기본 정리

먼저 하남교산과 공간적으로 연결되는 도로 링크를 기준으로  
시설 분류에 필요한 기본 속성을 정리한다.

In [24]:
facility_base = road_gyosan.copy()

use_cols = [
    "road_name", "road_rank_label", "road_type_label",
    "link_type_label", "oneway_label", "blockType", "geometry"
]
use_cols = [c for c in use_cols if c in facility_base.columns]

facility_base = facility_base[use_cols].copy()
facility_base = facility_base.drop_duplicates()

print(facility_base.shape)
display(facility_base.head())

(140, 7)


,road_name,road_rank_label,road_type_label,link_type_label,oneway_label,blockType,geometry
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원,"LINESTRING (127.19025 37.49595, 127.19019 37.4..."
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),경관녹지,"LINESTRING (127.19025 37.49595, 127.19019 37.4..."
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공동주택,"LINESTRING (127.19025 37.49595, 127.19019 37.4..."
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),도로,"LINESTRING (127.19025 37.49595, 127.19019 37.4..."
16769,고골로,시군도,일반도로,비분리,규제없음(양방통행),도로,"LINESTRING (127.19191 37.50363, 127.19176 37.5..."


## 2. 시설 유형별 후보지 분류 규칙

하남교산의 토지이용과 도로 특성을 함께 반영하여  
시설 후보지를 다음과 같이 분류한다.

In [25]:
child_blocks = {
    "교육시설", "공동주택", "단독주택", "공원", "보행자전용도로", "주상복합"
}

elderly_blocks = {
    "근린생활시설용지", "상업시설", "종합의료시설", "공공공지",
    "사회복지시설", "주차장", "공공청사"
}

general_blocks = {
    "도로", "공동주택", "단독주택", "주상복합", "근린생활시설용지", "상업시설"
}

In [26]:
facility_base["is_general_road"] = (facility_base["road_type_label"] == "일반도로").astype(int)
facility_base["is_two_way"] = (facility_base["oneway_label"] == "규제없음(양방통행)").astype(int)

facility_base["child_candidate"] = (
    facility_base["blockType"].isin(child_blocks) &
    (facility_base["is_general_road"] == 1)
).astype(int)

facility_base["elderly_candidate"] = (
    facility_base["blockType"].isin(elderly_blocks) &
    (facility_base["is_general_road"] == 1)
).astype(int)

facility_base["speed_candidate"] = (
    facility_base["blockType"].isin(general_blocks) &
    (facility_base["is_general_road"] == 1) &
    (facility_base["is_two_way"] == 1)
).astype(int)

In [27]:
print("어린이형 후보지 수:", facility_base["child_candidate"].sum())
print("노인형 후보지 수:", facility_base["elderly_candidate"].sum())
print("속도관리형 후보지 수:", facility_base["speed_candidate"].sum())

어린이형 후보지 수: 39
노인형 후보지 수: 13
속도관리형 후보지 수: 69


## 3. 시설 후보지 예시 확인

분류된 후보지가 실제로 어떤 토지이용 및 도로 속성을 가지는지 확인한다.

In [28]:
print("=== 어린이형 후보지 예시 ===")
display(
    facility_base.loc[facility_base["child_candidate"] == 1,
                      ["road_name", "blockType", "road_rank_label", "road_type_label", "link_type_label", "oneway_label"]]
    .head(20)
)

print("=== 노인형 후보지 예시 ===")
display(
    facility_base.loc[facility_base["elderly_candidate"] == 1,
                      ["road_name", "blockType", "road_rank_label", "road_type_label", "link_type_label", "oneway_label"]]
    .head(20)
)

print("=== 속도관리형 후보지 예시 ===")
display(
    facility_base.loc[facility_base["speed_candidate"] == 1,
                      ["road_name", "blockType", "road_rank_label", "road_type_label", "link_type_label", "oneway_label"]]
    .head(20)
)

=== 어린이형 후보지 예시 ===


,road_name,blockType,road_rank_label,road_type_label,link_type_label,oneway_label
16579,고골로,공원,시군도,일반도로,비분리,규제없음(양방통행)
16579,고골로,공동주택,시군도,일반도로,비분리,규제없음(양방통행)
16769,고골로,주상복합,시군도,일반도로,비분리,규제없음(양방통행)
16769,고골로,교육시설,시군도,일반도로,비분리,규제없음(양방통행)
17970,대성로,공원,시군도,일반도로,비분리,규제없음(양방통행)
18099,고골로,공원,시군도,일반도로,비분리,규제없음(양방통행)
18290,고골로,공원,시군도,일반도로,비분리,규제없음(양방통행)
18551,고골로,공원,시군도,일반도로,비분리,규제없음(양방통행)
18870,대성로,공원,시군도,일반도로,교차로 통로,규제없음(양방통행)
18871,고골로,공원,시군도,일반도로,교차로 통로,일방통행


=== 노인형 후보지 예시 ===


,road_name,blockType,road_rank_label,road_type_label,link_type_label,oneway_label
17970,대성로,상업시설,시군도,일반도로,비분리,규제없음(양방통행)
17970,대성로,공공청사,시군도,일반도로,비분리,규제없음(양방통행)
18731,대성로,공공청사,시군도,일반도로,비분리,규제없음(양방통행)
17998,샘재로,사회복지시설,시군도,일반도로,비분리,규제없음(양방통행)
18167,대성로,공공청사,시군도,일반도로,비분리,규제없음(양방통행)
18301,대성로,공공청사,시군도,일반도로,비분리,규제없음(양방통행)
18990,None,공공청사,시군도,일반도로,교차로 통로,일방통행
18991,대성로,공공청사,시군도,일반도로,교차로 통로,규제없음(양방통행)
18993,고골로,공공청사,시군도,일반도로,교차로 통로,규제없음(양방통행)
18188,고골로242번길,종합의료시설,시군도,일반도로,비분리/진출입로,규제없음(양방통행)


=== 속도관리형 후보지 예시 ===


,road_name,blockType,road_rank_label,road_type_label,link_type_label,oneway_label
16579,고골로,공동주택,시군도,일반도로,비분리,규제없음(양방통행)
16579,고골로,도로,시군도,일반도로,비분리,규제없음(양방통행)
16769,고골로,도로,시군도,일반도로,비분리,규제없음(양방통행)
17970,대성로,도로,시군도,일반도로,비분리,규제없음(양방통행)
17998,샘재로,도로,시군도,일반도로,비분리,규제없음(양방통행)
18100,서하남로,도로,시군도,일반도로,비분리,규제없음(양방통행)
18101,서하남로,도로,시군도,일반도로,비분리,규제없음(양방통행)
18112,고골로,도로,시군도,일반도로,비분리,규제없음(양방통행)
18188,고골로242번길,도로,시군도,일반도로,비분리/진출입로,규제없음(양방통행)
18232,None,도로,시군도,일반도로,비분리/진출입로,규제없음(양방통행)


## 4. 문헌 기반 효과계수 설정

본 연구에서는 실제 사전·사후 자료가 없으므로,
공식 문헌과 기관 자료에서 제시된 안전효과를 참고하여
시설별 시나리오 계수를 설정하였다.

- 어린이형: RRFB / 스마트 횡단보도 계열
- 노인형: PHB / 보행 신호 보강 계열
- 속도관리형: 속도 안전 카메라 / 동적 속도경고 계열

이 값은 직접적인 인과추정 결과가 아니라,
시설별 기대효과를 추정하기 위한 정책 시나리오 계수로 활용한다.

In [31]:
effect_table = pd.DataFrame({
    "facility_type": [
        "child_candidate",
        "elderly_candidate",
        "speed_candidate"
    ],
    "facility_name": [
        "RRFB / Smart Crosswalk",
        "PHB / Pedestrian Signal Support",
        "Speed Safety Management"
    ],
    "cmf": [
        0.526,   # RRFB 
        0.45,    # PHB
        0.80     # 
    ]
})

effect_table["crf"] = 1 - effect_table["cmf"]
display(effect_table)

,facility_type,facility_name,cmf,crf
0,child_candidate,RRFB / Smart Crosswalk,0.526,0.474
1,elderly_candidate,PHB / Pedestrian Signal Support,0.450,0.550
2,speed_candidate,Speed Safety Management,0.800,0.200


## 5. 기대 효과 산정 구조

후보지별 효과는 다음 논리로 산정한다.

- 기본 위험도: 하남교산 위험도 결과(`risk_score` 또는 `risk_prob`)
- 시설 적합성: 후보지 유형 일치 여부
- 문헌 기반 효과계수: CMF 또는 CRF

즉, 위험도가 높은 지점일수록,
그리고 시설 유형과 공간 맥락이 잘 맞을수록
기대 사고 저감 효과가 크게 나타나도록 설계한다.

## 6. 하남교산 최신 위험도 결과 불러오기

하남교산 위험도 결과는 가장 최근에 생성된 `v2` 버전을 우선 사용한다.  
특히 시설 후보지와 공간적으로 결합하기 위해 geometry가 포함된 `geojson` 파일을 우선 활용한다.

사용 우선순위는 다음과 같다.

1. `02_gyosan_risk_score_final_v2.geojson`
2. `02_gyosan_risk_score_final.geojson`
3. `02_gyosan_risk_score_final_v2.csv`
4. `02_gyosan_risk_score_final.csv`
5. `gyosan_risk_score.csv`

In [32]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

OUT_DIR = Path("../outputs/processed")

risk_candidates = [
    OUT_DIR / "02_gyosan_risk_score_final_v2.geojson",
    OUT_DIR / "02_gyosan_risk_score_final.geojson",
    OUT_DIR / "02_gyosan_risk_score_final_v2.csv",
    OUT_DIR / "02_gyosan_risk_score_final.csv",
    OUT_DIR / "gyosan_risk_score.csv"
]

risk_fp = None
for fp in risk_candidates:
    if fp.exists():
        risk_fp = fp
        break

print("selected risk file:", risk_fp)

selected risk file: ../outputs/processed/02_gyosan_risk_score_final_v2.geojson


In [34]:
if risk_fp is None:
    raise FileNotFoundError("하남교산 위험도 파일을 찾을 수 없습니다.")

if risk_fp.suffix.lower() == ".geojson":
    risk_df = gpd.read_file(risk_fp)
else:
    risk_df = pd.read_csv(risk_fp)

print(type(risk_df))
print("shape:", risk_df.shape)
print(risk_df.columns.tolist())

display(risk_df.head(3))

<class 'geopandas.geodataframe.GeoDataFrame'>
shape: (770, 31)
['gid', 'city', 'resident_pop', 'flow_pop', 'working_pop', 'visit_pop', 'service_pop', 'mean_speed', 'traffic_volume', 'congestion_freq', 'congestion_time', 'crosswalk_count', 'bus_stop_count', 'school_count', 'kinder_count', 'daycare_count', 'cctv_count', 'speedbump_count', 'is_residential', 'is_commercial', 'is_green', 'is_road', 'is_public', 'blockType_main', 'landuse_group', 'risk_prob', 'risk_intensity', 'risk_score', 'risk_rank_pct', 'risk_grade', 'geometry']


,gid,city,resident_pop,flow_pop,working_pop,visit_pop,service_pop,mean_speed,traffic_volume,congestion_freq,...,is_road,is_public,blockType_main,landuse_group,risk_prob,risk_intensity,risk_score,risk_rank_pct,risk_grade,geometry
0,다사720443,경기도 하남시,47.607266,1.536856,0.488373,1.048483,0.126637,10.657618,129.747785,12.678988,...,0,0,unknown,other,0.191250,1.294472,0.247567,0.136364,Low,"POLYGON ((127.18323 37.49754, 127.18323 37.498..."
1,다사720444,경기도 하남시,47.607266,4.423908,1.056933,3.366975,0.367688,19.740852,242.285221,52.873690,...,0,0,단독주택,other,0.685228,1.591526,1.090558,0.694156,Medium,"POLYGON ((127.18323 37.49844, 127.18323 37.499..."
2,다사721443,경기도 하남시,47.607266,1.536856,0.488373,1.048483,0.126637,10.657618,129.747785,12.678988,...,0,0,unknown,other,0.191250,1.294472,0.247567,0.136364,Low,"POLYGON ((127.18436 37.49754, 127.18436 37.498..."


## 7. 위험도 데이터 구조 확인

위험도 결과 파일에서 다음 정보를 우선 확인한다.

- 공간정보(geometry) 포함 여부
- 격자 식별자(`gid`) 존재 여부
- 위험도 컬럼명 (`risk_score`, `risk_prob`, `risk_intensity` 등)

이후 가장 대표적인 위험도 지표를 선택하여 시설 후보지와 연결한다.

In [35]:
risk_cols = [c for c in ["risk_score", "risk_prob", "risk_intensity", "risk_rank_pct", "risk_grade"] if c in risk_df.columns]
print("usable risk columns:", risk_cols)

if "risk_score" in risk_df.columns:
    main_risk_col = "risk_score"
elif "risk_prob" in risk_df.columns:
    main_risk_col = "risk_prob"
elif "risk_intensity" in risk_df.columns:
    main_risk_col = "risk_intensity"
else:
    raise ValueError("사용 가능한 대표 위험도 컬럼을 찾지 못했습니다.")

print("selected main risk column:", main_risk_col)

usable risk columns: ['risk_score', 'risk_prob', 'risk_intensity', 'risk_rank_pct', 'risk_grade']
selected main risk column: risk_score


## 8. 시설 후보지와 위험도 결과 연결

시설 후보지는 도로 링크 단위, 위험도 결과는 격자 단위이므로  
가능하면 공간 결합(spatial join)을 우선 적용한다.

만약 위험도 결과가 CSV 형식으로 geometry가 없을 경우에는  
`gid`를 기준으로 별도 결합이 필요하다.

In [36]:
candidate_gdf = facility_base.copy()

# GeoDataFrame 보장
if not isinstance(candidate_gdf, gpd.GeoDataFrame):
    candidate_gdf = gpd.GeoDataFrame(candidate_gdf, geometry="geometry", crs=road_gyosan.crs)

# geometry 포함 여부 확인
is_risk_geo = isinstance(risk_df, gpd.GeoDataFrame) and "geometry" in risk_df.columns

print("is_risk_geo:", is_risk_geo)

is_risk_geo: True


In [37]:
if is_risk_geo:
    risk_gdf = risk_df.copy().to_crs(candidate_gdf.crs)

    candidate_risk = gpd.sjoin(
        candidate_gdf.drop(columns=[c for c in ["index_left", "index_right"] if c in candidate_gdf.columns], errors="ignore"),
        risk_gdf[[main_risk_col, "geometry"]],
        how="left",
        predicate="intersects"
    ).drop(columns=["index_right"], errors="ignore")

else:
    candidate_risk = candidate_gdf.copy()
    candidate_risk[main_risk_col] = np.nan

print(candidate_risk.shape)
display(candidate_risk.head(3))

(782, 13)


,road_name,road_rank_label,road_type_label,link_type_label,oneway_label,blockType,geometry,is_general_road,is_two_way,child_candidate,elderly_candidate,speed_candidate,risk_score
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원,"LINESTRING (127.19025 37.49595, 127.19019 37.4...",1,1,1,0,0,0.469310
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원,"LINESTRING (127.19025 37.49595, 127.19019 37.4...",1,1,1,0,0,0.337568
16579,고골로,시군도,일반도로,비분리,규제없음(양방통행),공원,"LINESTRING (127.19025 37.49595, 127.19019 37.4...",1,1,1,0,0,0.487252


## 9. 시설 유형별 효과계수 설정

직접적인 사전-사후 관측자료가 없으므로,  
공식 문헌과 기관 자료에서 제시된 안전 효과를 참고하여 시설별 효과계수를 설정한다.

본 연구에서는 다음과 같이 가정한다.

- 어린이 보행 안전형: RRFB / Smart Crosswalk 계열
- 노인 생활이동 안전형: PHB / 보행 신호 보강 계열
- 일반 생활도로 속도관리형: Speed Safety Management 계열

여기서 CMF는 사고 후 기대 발생 비율,  
CRF는 사고 감소 비율(= 1 - CMF)이다.

In [38]:
effect_table = pd.DataFrame({
    "facility_type": ["child_candidate", "elderly_candidate", "speed_candidate"],
    "facility_name": [
        "RRFB / Smart Crosswalk",
        "PHB / Pedestrian Signal Support",
        "Speed Safety Management"
    ],
    "cmf": [
        0.526,   # RRFB 사례
        0.45,    # PHB 사례
        0.80     # 속도관리 보수적 시나리오
    ]
})

effect_table["crf"] = 1 - effect_table["cmf"]
display(effect_table)

,facility_type,facility_name,cmf,crf
0,child_candidate,RRFB / Smart Crosswalk,0.526,0.474
1,elderly_candidate,PHB / Pedestrian Signal Support,0.450,0.550
2,speed_candidate,Speed Safety Management,0.800,0.200


## 10. 후보지 유형 부여

하남교산 도로 링크를 토지이용과 도로 속성에 따라  
어린이형, 노인형, 속도관리형 후보지로 구분한다.

In [39]:
candidate_risk["facility_type"] = np.select(
    [
        candidate_risk["child_candidate"] == 1,
        candidate_risk["elderly_candidate"] == 1,
        candidate_risk["speed_candidate"] == 1
    ],
    [
        "child_candidate",
        "elderly_candidate",
        "speed_candidate"
    ],
    default="none"
)

candidate_risk = candidate_risk.merge(
    effect_table,
    on="facility_type",
    how="left"
)

display(candidate_risk[[
    c for c in ["road_name", "blockType", "facility_type", "facility_name", main_risk_col, "cmf", "crf"]
    if c in candidate_risk.columns
]].head(10))

,road_name,blockType,facility_type,facility_name,risk_score,cmf,crf
0,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.469310,0.526,0.474
1,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.337568,0.526,0.474
2,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.487252,0.526,0.474
3,고골로,공원,child_candidate,RRFB / Smart Crosswalk,1.856564,0.526,0.474
4,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.526,0.474
5,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.526,0.474
6,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.526,0.474
7,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.589500,0.526,0.474
8,고골로,경관녹지,none,NaN,0.469310,NaN,NaN
9,고골로,경관녹지,none,NaN,0.337568,NaN,NaN


## 11. 기대 사고 저감 효과 산정

후보지별 기대 효과는 다음과 같은 시나리오 방식으로 계산한다.

- 기본 위험도: 하남교산 위험도 결과
- 시설 효과: 문헌 기반 CRF
- 기대 저감 점수: `위험도 × CRF`

이는 실제 사고 건수 감소를 직접 관측한 결과가 아니라,  
시설 설치 시 상대적으로 얼마나 큰 안전 개선 효과가 기대되는지 비교하기 위한 지표이다.

In [40]:
candidate_risk["expected_reduction_score"] = candidate_risk[main_risk_col] * candidate_risk["crf"]

summary_cols = [c for c in [
    "road_name", "blockType", "facility_type", "facility_name",
    main_risk_col, "crf", "expected_reduction_score"
] if c in candidate_risk.columns]

display(candidate_risk[summary_cols].head(10))

,road_name,blockType,facility_type,facility_name,risk_score,crf,expected_reduction_score
0,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.469310,0.474,0.222453
1,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.337568,0.474,0.160007
2,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.487252,0.474,0.230958
3,고골로,공원,child_candidate,RRFB / Smart Crosswalk,1.856564,0.474,0.880011
4,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.474,0.324822
5,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.474,0.324822
6,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.685279,0.474,0.324822
7,고골로,공원,child_candidate,RRFB / Smart Crosswalk,0.589500,0.474,0.279423
8,고골로,경관녹지,none,NaN,0.469310,NaN,NaN
9,고골로,경관녹지,none,NaN,0.337568,NaN,NaN


## 12. 유형별 우선 설치 후보지 도출

기대 저감 점수를 기준으로 유형별 상위 후보지를 정렬한다.  
이 결과는 실제 설계 도면이 아니라, 우선적으로 검토할 필요가 있는 설치 대상지를 제안하는 분석 결과이다.

In [41]:
top_child = candidate_risk.loc[candidate_risk["facility_type"] == "child_candidate"].sort_values(
    "expected_reduction_score", ascending=False
).head(10)

top_elderly = candidate_risk.loc[candidate_risk["facility_type"] == "elderly_candidate"].sort_values(
    "expected_reduction_score", ascending=False
).head(10)

top_speed = candidate_risk.loc[candidate_risk["facility_type"] == "speed_candidate"].sort_values(
    "expected_reduction_score", ascending=False
).head(10)

print("=== Top child candidates ===")
display(top_child[summary_cols])

print("=== Top elderly candidates ===")
display(top_elderly[summary_cols])

print("=== Top speed-management candidates ===")
display(top_speed[summary_cols])

=== Top child candidates ===


,road_name,blockType,facility_type,facility_name,risk_score,crf,expected_reduction_score
281,고골로,공원,child_candidate,RRFB / Smart Crosswalk,1.988850,0.474,0.942715
471,샘재로,주상복합,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
511,샘재로,교육시설,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
531,샘재로,단독주택,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
451,샘재로,공원,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
279,고골로,공원,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
431,샘재로,공동주택,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
491,샘재로,보행자전용도로,child_candidate,RRFB / Smart Crosswalk,1.986944,0.474,0.941811
635,대성로,공원,child_candidate,RRFB / Smart Crosswalk,1.969097,0.474,0.933352
505,샘재로,교육시설,child_candidate,RRFB / Smart Crosswalk,1.956878,0.474,0.927560


=== Top elderly candidates ===


,road_name,blockType,facility_type,facility_name,risk_score,crf,expected_reduction_score
551,샘재로,사회복지시설,elderly_candidate,PHB / Pedestrian Signal Support,1.986944,0.55,1.092819
293,대성로,상업시설,elderly_candidate,PHB / Pedestrian Signal Support,1.956878,0.55,1.076283
356,대성로,공공청사,elderly_candidate,PHB / Pedestrian Signal Support,1.956878,0.55,1.076283
545,샘재로,사회복지시설,elderly_candidate,PHB / Pedestrian Signal Support,1.956878,0.55,1.076283
295,대성로,상업시설,elderly_candidate,PHB / Pedestrian Signal Support,1.944973,0.55,1.069735
358,대성로,공공청사,elderly_candidate,PHB / Pedestrian Signal Support,1.944973,0.55,1.069735
549,샘재로,사회복지시설,elderly_candidate,PHB / Pedestrian Signal Support,1.940985,0.55,1.067542
550,샘재로,사회복지시설,elderly_candidate,PHB / Pedestrian Signal Support,1.908108,0.55,1.049460
537,샘재로,사회복지시설,elderly_candidate,PHB / Pedestrian Signal Support,1.894206,0.55,1.041814
302,대성로,상업시설,elderly_candidate,PHB / Pedestrian Signal Support,1.869862,0.55,1.028424


=== Top speed-management candidates ===


,road_name,blockType,facility_type,facility_name,risk_score,crf,expected_reduction_score
105,고골로,도로,speed_candidate,Speed Safety Management,1.988850,0.2,0.397770
109,서하남로,도로,speed_candidate,Speed Safety Management,1.988850,0.2,0.397770
82,None,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
67,샘재로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
103,고골로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
106,서하남로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
86,서하남로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
85,서하남로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
108,서하남로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389
83,서하남로,도로,speed_candidate,Speed Safety Management,1.986944,0.2,0.397389


## 14. 적용 시설 설명

### RRFB / Smart Crosswalk
보행자 접근 또는 횡단 시 차량에게 강한 시각적 경고를 제공하는 시설이다.  
교육시설, 공동주택, 공원 인접 구간과 같이 보행 수요가 높은 생활권 일반도로에 적합하다.

### PHB / Pedestrian Signal Support
보행자의 횡단 요청 또는 감지에 따라 차량 정지를 유도하는 보행 안전 시설이다.  
상업시설, 근린생활시설, 의료시설 등 생활 이동 수요가 높은 구간에 적합하다.

### Speed Safety Management
동적 속도경고, 속도 안전 카메라, 감속 유도 표지 등으로 구성되는 속도관리형 시설이다.  
양방통행 일반도로와 주거·상업 혼합 생활권에서 효과적이다.

## 15. 우선 설치 후보지 결과 저장

유형별 기대 저감 점수를 기준으로 상위 후보지를 추출한 뒤,  
QGIS에서 바로 확인할 수 있도록 GeoJSON 파일로 저장한다.

저장 파일은 다음과 같다.

- 어린이형 후보지
- 노인형 후보지
- 속도관리형 후보지
- 전체 후보지(참고용)

In [44]:
from pathlib import Path

OUT_DIR = Path("../outputs/processed")
QGIS_DIR = OUT_DIR / "qgis_outputs"
QGIS_DIR.mkdir(parents=True, exist_ok=True)

save_cols = [c for c in [
    "road_name", "blockType", "facility_type", "facility_name",
    "risk_score", "cmf", "crf", "expected_reduction_score", "geometry"
] if c in candidate_risk.columns]

child_save = top_child[save_cols].copy()
elderly_save = top_elderly[save_cols].copy()
speed_save = top_speed[save_cols].copy()
all_save = candidate_risk[save_cols].copy()

child_fp = QGIS_DIR / "gyosan_top_child_facility_candidates.geojson"
elderly_fp = QGIS_DIR / "gyosan_top_elderly_facility_candidates.geojson"
speed_fp = QGIS_DIR / "gyosan_top_speed_facility_candidates.geojson"
all_fp = QGIS_DIR / "gyosan_all_facility_candidates.geojson"

child_save.to_file(child_fp, driver="GeoJSON")
elderly_save.to_file(elderly_fp, driver="GeoJSON")
speed_save.to_file(speed_fp, driver="GeoJSON")
all_save.to_file(all_fp, driver="GeoJSON")

print("saved:")
print(child_fp)
print(elderly_fp)
print(speed_fp)
print(all_fp)

saved:
../outputs/processed/qgis_outputs/gyosan_top_child_facility_candidates.geojson
../outputs/processed/qgis_outputs/gyosan_top_elderly_facility_candidates.geojson
../outputs/processed/qgis_outputs/gyosan_top_speed_facility_candidates.geojson
../outputs/processed/qgis_outputs/gyosan_all_facility_candidates.geojson


## 16. 중복 후보지 정리

동일 도로명 또는 동일 링크가 여러 토지이용 polygon과 동시에 교차할 수 있으므로,  
기대 저감 점수가 가장 높은 항목을 기준으로 중복을 정리한 최종 추천안도 함께 생성한다.

In [45]:
def make_final_recommendation(df):
    temp = df.copy()
    key_col = "road_name" if "road_name" in temp.columns else temp.index
    temp["road_name"] = temp["road_name"].fillna("Unnamed_Road")
    temp = temp.sort_values("expected_reduction_score", ascending=False)
    temp = temp.drop_duplicates(subset=["road_name"])
    return temp

child_final = make_final_recommendation(top_child)
elderly_final = make_final_recommendation(top_elderly)
speed_final = make_final_recommendation(top_speed)

child_final_fp = QGIS_DIR / "gyosan_final_child_facility_recommendation.geojson"
elderly_final_fp = QGIS_DIR / "gyosan_final_elderly_facility_recommendation.geojson"
speed_final_fp = QGIS_DIR / "gyosan_final_speed_facility_recommendation.geojson"

child_final[save_cols].to_file(child_final_fp, driver="GeoJSON")
elderly_final[save_cols].to_file(elderly_final_fp, driver="GeoJSON")
speed_final[save_cols].to_file(speed_final_fp, driver="GeoJSON")

print("final recommendation saved:")
print(child_final_fp)
print(elderly_final_fp)
print(speed_final_fp)

final recommendation saved:
../outputs/processed/qgis_outputs/gyosan_final_child_facility_recommendation.geojson
../outputs/processed/qgis_outputs/gyosan_final_elderly_facility_recommendation.geojson
../outputs/processed/qgis_outputs/gyosan_final_speed_facility_recommendation.geojson


In [46]:
child_final.drop(columns="geometry").to_csv(
    QGIS_DIR / "gyosan_final_child_facility_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)

elderly_final.drop(columns="geometry").to_csv(
    QGIS_DIR / "gyosan_final_elderly_facility_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)

speed_final.drop(columns="geometry").to_csv(
    QGIS_DIR / "gyosan_final_speed_facility_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)

#QGIS용

## 17. 문헌 기반 효과계수를 적용한 우선 설치 후보지 도출 결과

하남교산의 최신 위험도 결과(`02_gyosan_risk_score_final_v2`)와  
토지이용계획도-도로망 연계 결과를 결합하여,  
시설 유형별 기대 저감 점수를 산정하였다.

기대 저감 점수는 다음과 같이 계산하였다.

- 기본 위험도: `risk_score`
- 시설 효과계수: 문헌 기반 CRF
- 기대 저감 점수: `risk_score × CRF`

본 결과는 실제 사전·사후 관측을 통한 인과효과 추정이 아니라,  
하남교산의 공간 맥락에 적합한 시설을 적용했을 때의  
상대적 기대 효과를 비교한 시나리오 기반 결과이다.

### 17.1 어린이형 시설 후보지

어린이형 후보지는 공원, 교육시설, 공동주택, 단독주택, 보행자전용도로, 주상복합 인접 구간을 중심으로 도출되었다.

상위 후보지는 주로 `고골로`, `샘재로`, `대성로` 일대에 집중되었으며,  
특히 공원·교육시설·주거 블록과 접하는 일반도로 구간에서 높은 위험 점수가 확인되었다.

이는 하남교산에서 어린이 보행사고 위험이  
단순 통학시설 주변에만 국한되지 않고,  
주거-공원-교육 기능이 결합된 생활권 일반도로 전반에 존재할 수 있음을 의미한다.

따라서 어린이형 시설은  
단일 지점 대응보다 생활권 보행축 단위로 설치를 검토할 필요가 있다.

### 17.2 노인형 시설 후보지

노인형 후보지는 사회복지시설, 상업시설, 공공청사 등  
생활이동 목적지가 위치한 블록 인접 도로에서 높게 나타났다.

상위 후보지는 `샘재로`, `대성로` 일대에 집중되었으며,  
이는 노인 사고가 단순 보행시설 부족보다는  
생활 서비스 접근 구간과 교통량·유동인구가 높은 일반도로에서 발생할 가능성이 크다는 점을 시사한다.

따라서 노인형 시설은  
보행 횡단 지원, 보행 신호 보강, 시인성 강화, 경고 시스템과 같이  
생활이동 안전 확보에 초점을 맞춘 대책이 우선적으로 필요하다.

### 17.3 속도관리형 시설 후보지

속도관리형 후보지는 `도로` blockType에 해당하는 일반도로 구간을 중심으로 도출되었으며,  
특히 `고골로`, `서하남로`, `샘재로` 등 주요 생활도로 축에서 높은 위험 점수가 확인되었다.

이 구간들은 대부분 양방통행의 일반도로로 해석되었고,  
생활권 내부에서 차량과 보행이 혼재될 가능성이 높다.

따라서 속도관리형 시설은  
동적 속도경고, 속도 안전 카메라, 감속 유도 표지와 같이  
운전자 감속을 직접 유도하는 방식이 적합하다.

## 18. 종합 해석

하남교산의 도로망은 대부분 시군도급 일반도로, 비분리, 양방통행 구조로 나타났으며,  
이는 생활권 중심 신도시 도로 특성을 반영한다.

문헌 기반 효과계수를 적용한 결과,  
하남교산의 우선 설치 후보지는 크게 세 가지 축으로 정리된다.

1. 공원·교육시설·주거 인접 생활도로  
   → 어린이 보행 안전시설 우선 검토

2. 상업·사회복지·공공서비스 인접 생활도로  
   → 노인 생활이동 안전시설 우선 검토

3. 주요 일반도로 축  
   → 속도관리형 시설 우선 검토

즉, 하남교산에서는 고속 간선형 대책보다  
생활권 일반도로와 보행 활동 구간에 특화된 스마트 안전시설을 우선 배치하는 것이 합리적이다.

## 19. 결과 활용 방안

최종적으로 저장한 GeoJSON 파일은 QGIS에서 불러와  
다음과 같은 방식으로 활용할 수 있다.

- 유형별 후보지 시각화
- 기대 저감 점수 기준 graduated symbol 적용
- 토지이용계획도와 중첩 시각화
- 최종 보고서 및 발표 자료의 위치도 작성

이를 통해 단순 위험격자 제시를 넘어,  
실제 도로 링크 단위의 시설 설치 검토 결과를 시각적으로 제시할 수 있다.

In [48]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("../outputs/processed")
REF_DIR = OUT_DIR / "references"
REF_DIR.mkdir(parents=True, exist_ok=True)

facility_refs = [
    {
        "facility_type": "child_candidate",
        "facility_name": "RRFB / Smart Crosswalk",
        "source_type": "Official CMF / FHWA",
        "title": "Install rectangular rapid flashing beacon (RRFB)",
        "link": "https://cmfclearinghouse.fhwa.dot.gov/detail.php?facid=9024",
        "meaning": "A pedestrian crossing treatment that uses rapid flashing LED beacons to alert drivers when pedestrians are present or likely to cross.",
        "use_when": "Useful at unsignalized crossings, school access roads, park-front roads, and other pedestrian-priority crossings where driver yielding needs to be improved.",
        "why_child": "Children are more exposed to crossing-related crashes and have lower speed judgment. RRFB-type treatments are appropriate where school, park, and residential walking trips overlap.",
        "why_elderly": "Helpful for elderly pedestrians as well, especially where longer decision time and better driver attention are needed at crossings.",
        "why_speed_zone": "It is not primarily a speed control device, but it can indirectly lower approach speed by increasing driver awareness near crossings.",
        "key_effect_note": "CMF example: 0.526 (CRF 47.4%) for vehicle-pedestrian crashes."
    },
    {
        "facility_type": "elderly_candidate",
        "facility_name": "PHB / Pedestrian Signal Support",
        "source_type": "Official FHWA",
        "title": "Pedestrian Hybrid Beacons",
        "link": "https://highways.dot.gov/safety/proven-safety-countermeasures/pedestrian-hybrid-beacons",
        "meaning": "A pedestrian-activated traffic control device that stops vehicles and provides a protected crossing opportunity where a full signal may not be warranted.",
        "use_when": "Useful at higher-risk pedestrian crossings, uncontrolled intersections, and major walking corridors serving commercial, welfare, and medical destinations.",
        "why_child": "Useful where children need a stronger protected crossing treatment than warning-only systems.",
        "why_elderly": "Especially suitable for elderly movement corridors because it provides a clearer crossing opportunity and reduces decision burden at busy roads.",
        "why_speed_zone": "It affects vehicle operations more directly than warning-only systems, so it is more appropriate at key crossings than along every road segment.",
        "key_effect_note": "FHWA presents PHB as a proven pedestrian safety countermeasure."
    },
    {
        "facility_type": "speed_candidate",
        "facility_name": "Speed Safety Management",
        "source_type": "Official FHWA",
        "title": "Speed Safety Cameras",
        "link": "https://highways.dot.gov/safety/proven-safety-countermeasures/speed-safety-cameras",
        "meaning": "A speed management treatment that uses automated speed detection and enforcement or dynamic speed warning to reduce excessive speeding.",
        "use_when": "Useful on general two-way roads, high-risk approach segments, and mixed traffic environments where speeding contributes to crash severity.",
        "why_child": "Effective around child activity zones because lower vehicle speed directly improves survivability in pedestrian crashes.",
        "why_elderly": "Important in elderly activity areas because elderly pedestrians are more vulnerable to injury severity even at moderate speeds.",
        "why_speed_zone": "This is the most directly relevant treatment for speed-related risk because it targets speeding behavior itself.",
        "key_effect_note": "FHWA treats speed safety cameras as a proven safety countermeasure within a broader speed management program."
    },
    {
        "facility_type": "supporting_reference",
        "facility_name": "Pedestrian Countermeasure Toolbox",
        "source_type": "Official FHWA",
        "title": "Toolbox of Pedestrian Countermeasures and Their Potential Effectiveness",
        "link": "https://highways.dot.gov/safety/pedestrian-bicyclist/safety-tools/toolbox-pedestrian-countermeasures-and-their-potential",
        "meaning": "A reference document summarizing crash reduction factors for a wide range of pedestrian safety treatments.",
        "use_when": "Useful when selecting or comparing pedestrian safety interventions for crossings, intersections, medians, signals, and visibility improvements.",
        "why_child": "Supports the choice of school-zone and crossing-related pedestrian treatments.",
        "why_elderly": "Supports the choice of crossing support and signal-related treatments for slower or mobility-limited pedestrians.",
        "why_speed_zone": "Useful as a supporting reference when speed management is combined with pedestrian crossing treatments.",
        "key_effect_note": "Provides CRF-oriented evidence across multiple pedestrian countermeasures."
    },
    {
        "facility_type": "supporting_reference",
        "facility_name": "STEP Resources",
        "source_type": "Official FHWA",
        "title": "STEP Resources",
        "link": "https://highways.dot.gov/safety/pedestrian-bicyclist/step/resources",
        "meaning": "FHWA resource hub for pedestrian safety countermeasures, including PHB, RRFB, raised crosswalks, road diets, and visibility enhancements.",
        "use_when": "Useful when building the rationale section of a report or when selecting context-appropriate pedestrian safety treatments.",
        "why_child": "Helpful for identifying school and neighborhood pedestrian safety treatments.",
        "why_elderly": "Helpful for selecting treatments that improve crossing clarity, visibility, and protection for slower pedestrians.",
        "why_speed_zone": "Supports integrated planning when pedestrian risk and speed management need to be addressed together.",
        "key_effect_note": "Good umbrella reference for connecting multiple pedestrian safety facilities in one framework."
    }
]

ref_df = pd.DataFrame(facility_refs)
display(ref_df)

csv_fp = REF_DIR / "facility_reference_table.csv"
ref_df.to_csv(csv_fp, index=False, encoding="utf-8-sig")

md_lines = []
md_lines.append("# Facility Reference Notes\n")

for _, row in ref_df.iterrows():
    md_lines.append(f"## {row['facility_name']}")
    md_lines.append(f"- **Type**: {row['facility_type']}")
    md_lines.append(f"- **Source**: {row['source_type']}")
    md_lines.append(f"- **Reference title**: {row['title']}")
    md_lines.append(f"- **Link**: {row['link']}")
    md_lines.append(f"- **Meaning**: {row['meaning']}")
    md_lines.append(f"- **Useful when**: {row['use_when']}")
    md_lines.append(f"- **Why useful for child zones**: {row['why_child']}")
    md_lines.append(f"- **Why useful for elderly zones**: {row['why_elderly']}")
    md_lines.append(f"- **Why useful for speed-management zones**: {row['why_speed_zone']}")
    md_lines.append(f"- **Key effect note**: {row['key_effect_note']}")
    md_lines.append("")

md_fp = REF_DIR / "facility_reference_notes.md"
md_fp.write_text("\n".join(md_lines), encoding="utf-8")

print("saved:")
print(csv_fp)
print(md_fp)

,facility_type,facility_name,source_type,title,link,meaning,use_when,why_child,why_elderly,why_speed_zone,key_effect_note
0,child_candidate,RRFB / Smart Crosswalk,Official CMF / FHWA,Install rectangular rapid flashing beacon (RRFB),https://cmfclearinghouse.fhwa.dot.gov/detail.p...,A pedestrian crossing treatment that uses rapi...,"Useful at unsignalized crossings, school acces...",Children are more exposed to crossing-related ...,"Helpful for elderly pedestrians as well, espec...","It is not primarily a speed control device, bu...",CMF example: 0.526 (CRF 47.4%) for vehicle-ped...
1,elderly_candidate,PHB / Pedestrian Signal Support,Official FHWA,Pedestrian Hybrid Beacons,https://highways.dot.gov/safety/proven-safety-...,A pedestrian-activated traffic control device ...,"Useful at higher-risk pedestrian crossings, un...",Useful where children need a stronger protecte...,Especially suitable for elderly movement corri...,It affects vehicle operations more directly th...,FHWA presents PHB as a proven pedestrian safet...
2,speed_candidate,Speed Safety Management,Official FHWA,Speed Safety Cameras,https://highways.dot.gov/safety/proven-safety-...,A speed management treatment that uses automat...,"Useful on general two-way roads, high-risk app...",Effective around child activity zones because ...,Important in elderly activity areas because el...,This is the most directly relevant treatment f...,FHWA treats speed safety cameras as a proven s...
3,supporting_reference,Pedestrian Countermeasure Toolbox,Official FHWA,Toolbox of Pedestrian Countermeasures and Thei...,https://highways.dot.gov/safety/pedestrian-bic...,A reference document summarizing crash reducti...,Useful when selecting or comparing pedestrian ...,Supports the choice of school-zone and crossin...,Supports the choice of crossing support and si...,Useful as a supporting reference when speed ma...,Provides CRF-oriented evidence across multiple...
4,supporting_reference,STEP Resources,Official FHWA,STEP Resources,https://highways.dot.gov/safety/pedestrian-bic...,FHWA resource hub for pedestrian safety counte...,Useful when building the rationale section of ...,Helpful for identifying school and neighborhoo...,Helpful for selecting treatments that improve ...,Supports integrated planning when pedestrian r...,Good umbrella reference for connecting multipl...


saved:
../outputs/processed/references/facility_reference_table.csv
../outputs/processed/references/facility_reference_notes.md
